In [1]:
import pyarrow.dataset as ds
import pandas as pd
from pathlib import Path
import numpy as np
import statsmodels.api as sm
import os
import sys
from tqdm import tqdm

In [2]:
ROOT = Path("/home/hanwenying")
DATA = ROOT / "rothman-data/gltd"
OUT = ROOT / "rothman-anna/gltd/out"

In [3]:
id = int(os.environ.get("SLURM_ARRAY_TASK_ID", 0))

In [4]:
gtex = ds.dataset(DATA / "tpm-gtex-v11.parquet", format="parquet")

In [5]:
genelengths = pd.read_csv(DATA / "gencode-v47-lengths.tsv", sep='\t')
phenotypes = pd.read_csv(DATA / "phenotypes-gtex-v11.txt", sep='\t')
attributes = pd.read_csv(DATA / "attributes-gtex-v11.txt", sep='\t')

/tmp/ipykernel_203463/2919796179.py:3: DtypeWarning: Columns (0: SMGTC) have mixed types. Specify dtype option on import or set low_memory=False.
  attributes = pd.read_csv(DATA / "attributes-gtex-v11.txt", sep='\t')


In [6]:
genelengths = pd.DataFrame(genelengths.set_index('gene')['merged'])
genelengths['logmerged'] = np.log10(genelengths['merged'])

In [7]:
attributes['SUBJID'] = attributes['SAMPID'].str.extract(r'(^GTEX-[^-]+)')
attributes = attributes.dropna(subset=['SUBJID'])

/tmp/ipykernel_203463/2096518487.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  attributes['SUBJID'] = attributes['SAMPID'].str.extract(r'(^GTEX-[^-]+)')


In [8]:
metadata = pd.merge(phenotypes, attributes[['SAMPID', 'SUBJID', 'SMTSD']], on='SUBJID').drop(columns=['DTHHRDY'])
validsamples = gtex.schema.names
metadata = metadata[metadata['SAMPID'].isin(validsamples)]
metadata['SEX'] = metadata['SEX'] - 1

agemap = {'20-29': 0, '30-39': 1, '40-49': 2, '50-59': 3, '60-69': 4, '70-79': 5}
metadata['AGE'] = metadata['AGE'].map(agemap)

In [9]:
tissues_list = list(set(metadata['SMTSD']))

In [11]:
for i in tissues_list:
	print(i)

Esophagus - Gastroesophageal Junction
Heart - Left Ventricle
Nerve - Tibial
Cervix - Endocervix
Kidney - Cortex
Esophagus - Mucosa
Skin - Sun Exposed (Lower leg)
Testis
Pancreas - Acini
Heart - Atrial Appendage
Uterus
Esophagus - Muscularis
Small Intestine - Terminal Ileum
Brain - Substantia nigra
Liver - Portal Tract
Brain - Frontal Cortex (BA9)
Ovary
Cells - Cultured fibroblasts
Brain - Nucleus accumbens (basal ganglia)
Adrenal Gland
Brain - Cortex
Small Intestine - Terminal Ileum - Mixed Cell
Small Intestine - Terminal Ileum - Lymphoid Aggregate
Brain - Putamen (basal ganglia)
Bladder
Artery - Aorta
Colon - Sigmoid
Liver
Prostate
Fallopian Tube
Cervix - Ectocervix
Brain - Cerebellar Hemisphere
Brain - Anterior cingulate cortex (BA24)
Brain - Caudate (basal ganglia)
Lung
Artery - Coronary
Pancreas
Colon - Transverse - Mucosa
Pancreas - Islets
Artery - Tibial
Skin - Not Sun Exposed (Suprapubic)
Pancreas - Mixed Cell
Breast - Mammary Tissue
Vagina
Spleen
Colon - Transverse
Whole Blood


In [115]:
tissue = tissues_list[id]

In [ ]:
# tissue = "Whole Blood"

In [116]:
tissuesamples = metadata[metadata['SMTSD'] == tissue]

In [117]:
samps_to_get = tissuesamples['SAMPID']

In [118]:
gtexdf = gtex.to_table(columns=list(samps_to_get) + ["Name"]).to_pandas()

In [119]:
gtexdf = gtexdf[gtexdf.index.isin(gtexdf.index.intersection(genelengths.index))]

In [120]:
tissue_exp = gtexdf[list(tissuesamples['SAMPID'])]
tissue_exp_log = np.log2(tissue_exp + 1)

In [ ]:
highexp = ((tissue_exp_log > 0.5).sum(axis=1)) > 0.2 * tissue_exp.shape[1]
tissue_expdf = tissue_exp_log.loc[highexp]

In [122]:
tissue_expdf

,GTEX-1117F-0726-SM-5GIEN,GTEX-111FC-0626-SM-5N9CU,GTEX-111VG-0326-SM-5GZX7,GTEX-111YS-0326-SM-5GZZ3,GTEX-1122O-0626-SM-5N9B9,GTEX-117XS-0726-SM-5H131,GTEX-117YW-0426-SM-5GZZZ,GTEX-117YX-0626-SM-5EGJI,GTEX-1192W-0226-SM-5EGGT,GTEX-1192X-0726-SM-5987R,...,GTEX-ZVT4-1326-SM-5NQ8E,GTEX-ZVZQ-0726-SM-51MR3,GTEX-ZXG5-0626-SM-5NQ85,GTEX-ZYFC-1026-SM-5GZX9,GTEX-ZYFD-2026-SM-5E459,GTEX-ZYFG-0526-SM-5GZXX,GTEX-ZYT6-0926-SM-5GIEM,GTEX-ZYVF-1826-SM-5E44F,GTEX-ZZPT-0926-SM-5GICZ,GTEX-ZZPU-1126-SM-5N9CW
Name,,,,,,,,,,,,,,,,,,,,,
ENSG00000310526.1,1.787482,1.816640,0.699634,0.434188,0.946020,0.958751,0.614564,0.971661,0.911385,0.715360,...,1.902561,1.785367,1.030190,1.289693,0.694322,0.621394,0.937904,1.203888,1.538697,1.080590
ENSG00000308579.1,1.458874,2.564011,0.950576,0.996609,1.950155,2.118601,0.681795,0.679389,1.735630,1.397113,...,3.150026,1.687797,1.181840,1.034694,2.460046,0.739416,1.856713,2.122234,1.844149,2.520814
ENSG00000268903.1,0.364376,0.215918,0.370177,0.326946,0.155204,0.000000,0.088357,0.135943,0.327938,0.000000,...,0.579262,0.182016,0.000000,0.150085,0.000000,0.000000,0.164275,0.336808,0.426241,0.904570
ENSG00000269981.1,1.506557,2.716099,1.288037,1.810537,1.592456,1.877857,0.457983,0.474752,1.814029,0.866522,...,2.338961,1.445441,1.074827,0.148076,2.204983,0.814822,1.990091,1.879301,1.704537,3.262592
ENSG00000310527.1,1.240972,1.596880,0.847002,0.668287,0.963251,1.146439,0.560312,0.538228,1.486655,0.776983,...,1.249209,0.876311,1.527642,0.326487,1.074429,0.623516,1.189657,0.654846,1.941761,0.714085
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000267793.1,0.000000,1.523909,1.167425,0.373528,0.000000,0.838472,0.102161,0.156787,0.860308,0.241112,...,0.000000,0.000000,0.542368,0.398833,1.214431,0.000000,1.131604,0.000000,0.414453,0.000000
ENSG00000260197.1,0.000000,2.221089,1.674215,1.507549,0.000000,1.469526,0.663336,1.225729,1.510716,1.322996,...,0.000000,0.000000,1.742258,1.042179,1.124613,0.032953,1.656868,0.000000,1.866855,0.032658
ENSG00000012817.16,0.112453,4.801854,4.010631,3.945697,0.013149,4.141705,2.563297,4.307858,4.081844,4.097703,...,0.037807,0.030942,4.024217,3.673392,4.202630,0.319914,4.265177,0.064782,4.436101,0.049834


In [108]:
tissue_summary = pd.DataFrame(columns=["GENE", 'LENGTH', 'BETA', 'T', 'P'])

In [123]:
for gene in tqdm(tissue_expdf.index):
	genesummary = {'GENE': gene,
		 	'LOGLENGTH': genelengths.loc[gene]['logmerged']}
	y = tissue_expdf.loc[gene]
	X = pd.DataFrame({'INTERCEPT': np.ones(len(tissuesamples)), 'AGE':tissuesamples['AGE'], 'SEX':tissuesamples['SEX'], 'SAMPID': tissuesamples['SAMPID']}).set_index('SAMPID')

	model = sm.OLS(y, X)
	res = model.fit()

	beta_age, t, p = res.params['AGE'], res.tvalues['AGE'], res.pvalues['AGE']
	
	genesummary['BETA'] = beta_age
	genesummary['T'] = t
	genesummary['P'] = p

	tissue_summary.loc[len(tissue_summary)] = genesummary

100%|██████████| 20625/20625 [09:14<00:00, 37.18it/s]


In [124]:
tissue_summary.to_csv(OUT / f"{tissue}.tsv", sep='\t', index=False)